# 04 — Ground AQ Providers (WAQI, IQAir, Ambee)

Fetches *current/nearest* air quality from external providers for each site in `configs/run.yml`.

Hardening included:
- IQAir exponential backoff on HTTP 429 (rate limit)
- Provenance: raw JSON saved per provider + manifest with request metadata

Outputs:
- `outputs/04_ground_aq_providers/raw/*.json`
- `outputs/04_ground_aq_providers/provider_calls.json`
- `outputs/04_ground_aq_providers/manifest.json`


In [ ]:
import os, json, hashlib, time, random
from pathlib import Path
from datetime import datetime, timezone
import requests
import yaml

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def env(name: str):
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def sha256_file(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def manifest_base(step: str, config_paths=None):
    m = {'step': step, 'created_at_utc': utc_now(), 'configs': [], 'requests': [], 'artifacts': []}
    for cp in (config_paths or []):
        if cp and cp.exists():
            m['configs'].append({'path': str(cp), 'sha256': sha256_file(cp)})
    return m

def add_artifact(m: dict, path: Path, kind: str, meta=None):
    if path.exists():
        item = {'kind': kind, 'path': str(path), 'sha256': sha256_file(path)}
        if meta:
            item['meta'] = meta
        m['artifacts'].append(item)

def record_request(m: dict, url: str, params: dict | None, r: requests.Response, out_path: Path):
    rec = {
        'ts_utc': utc_now(),
        'url': url,
        'params': params or {},
        'status': int(r.status_code),
        'ok': bool(r.ok),
        'content_type': r.headers.get('content-type')
    }
    if out_path.exists():
        rec['response_path'] = str(out_path)
        rec['response_sha256'] = sha256_file(out_path)
    m['requests'].append(rec)

def find_repo_root(start: Path) -> Path:
    cur = start
    for _ in range(12):
        if (cur/'configs').exists() and (cur/'notebooks').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

CWD = Path.cwd().resolve()
ROOT = find_repo_root(CWD)
CONFIGS = ROOT/'configs'
OUTPUTS = ROOT/'outputs'

print('UTC now:', utc_now())
print('CWD:', CWD)
print('ROOT:', ROOT)
print('CONFIGS exists:', CONFIGS.exists())
print('OUTPUTS exists:', OUTPUTS.exists())


In [ ]:
cfg_path = CONFIGS/'run.yml'
if not cfg_path.exists():
    raise FileNotFoundError(f"configs/run.yml not found at: {cfg_path}")
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
def normalize_sites(cfg: dict):
    sites_norm = []
    sites = cfg.get('sites')
    if isinstance(sites, list):
        for s in sites:
            sid = s.get('id') or s.get('site_id') or s.get('name')
            sites_norm.append({
                'site_id': sid,
                'name': s.get('name'),
                'lat': s.get('lat'),
                'lon': s.get('lon'),
                'role': (s.get('role') or 'context')
            })
    elif isinstance(sites, dict):
        for k, v in sites.items():
            sites_norm.append({
                'site_id': k,
                'name': v.get('name'),
                'lat': v.get('lat'),
                'lon': v.get('lon'),
                'role': (v.get('role') or 'context')
            })
    else:
        raise RuntimeError("run.yml has no usable 'sites' (expected list or dict)")

    controls = cfg.get('controls')
    if isinstance(controls, dict):
        for k, v in controls.items():
            sites_norm.append({
                'site_id': k,
                'name': v.get('name'),
                'lat': v.get('lat'),
                'lon': v.get('lon'),
                'role': 'control'
            })

    usable = [s for s in sites_norm if s.get('site_id') and s.get('lat') is not None and s.get('lon') is not None]
    return usable
sites = normalize_sites(cfg)
print('Sites usable:', len(sites))
for s in sites:
    print('-', s['site_id'], s.get('name'), s.get('lat'), s.get('lon'), s.get('role'))


In [ ]:
def save_response(path: Path, r: requests.Response):
    ct = (r.headers.get('content-type') or '').lower()
    if 'application/json' in ct:
        try:
            write_json(path, r.json())
            return
        except Exception:
            pass
    write_json(path, {
        'status': 'nonjson_or_parse_error',
        'http_status': int(r.status_code),
        'content_type': r.headers.get('content-type'),
        'text': r.text[:20000]
    })

def iqair_with_backoff(lat, lon, api_key, session: requests.Session, max_attempts=6, base_sleep=2.0, timeout=30):
    url = 'https://api.airvisual.com/v2/nearest_city'
    params = {'lat': lat, 'lon': lon, 'key': api_key}
    last = None
    for attempt in range(1, max_attempts+1):
        r = session.get(url, params=params, timeout=timeout)
        last = r
        if r.status_code != 429:
            return r
        retry_after = r.headers.get('Retry-After')
        if retry_after:
            try:
                sleep_s = float(retry_after)
            except ValueError:
                sleep_s = base_sleep * (2 ** (attempt-1))
        else:
            sleep_s = base_sleep * (2 ** (attempt-1))
        sleep_s = min(60.0, sleep_s * (0.75 + 0.5*random.random()))
        print(f"IQAir 429 (attempt {attempt}/{max_attempts}); sleeping {sleep_s:.1f}s")
        time.sleep(sleep_s)
    return last


In [ ]:
step = '04_ground_aq_providers'
out = OUTPUTS/step
raw = out/'raw'
raw.mkdir(parents=True, exist_ok=True)

waqi = env('WAQI_TOKEN')
iqair = env('IQ_AIR_QUALITY_KEY')
ambee = env('GETAMBEE_AIR_QUALITY_KEY')

SESSION = requests.Session()
manifest = manifest_base(step, [cfg_path])

results = []
for s in sites:
    site_id = s['site_id']
    lat = float(s['lat'])
    lon = float(s['lon'])

    if waqi:
        url = f"https://api.waqi.info/feed/geo:{lat};{lon}/"
        r = SESSION.get(url, params={'token': waqi}, timeout=30)
        p = raw / f"waqi_{site_id}.json"
        save_response(p, r)
        record_request(manifest, url, {'token': '***'}, r, p)
        results.append({'provider':'waqi','site_id':site_id,'status':int(r.status_code),'ok':bool(r.ok),'path':str(p)})

    if iqair:
        url = 'https://api.airvisual.com/v2/nearest_city'
        r = iqair_with_backoff(lat, lon, iqair, session=SESSION)
        p = raw / f"iqair_{site_id}.json"
        save_response(p, r)
        record_request(manifest, url, {'lat': lat, 'lon': lon, 'key': '***'}, r, p)
        results.append({'provider':'iqair','site_id':site_id,'status':int(r.status_code),'ok':bool(r.ok),'path':str(p)})

    if ambee:
        url = 'https://api.ambeedata.com/latest/by-lat-lng'
        r = SESSION.get(url, params={'lat': lat, 'lng': lon}, headers={'x-api-key': ambee}, timeout=30)
        p = raw / f"ambee_{site_id}.json"
        save_response(p, r)
        record_request(manifest, url, {'lat': lat, 'lng': lon}, r, p)
        results.append({'provider':'ambee','site_id':site_id,'status':int(r.status_code),'ok':bool(r.ok),'path':str(p)})

    time.sleep(1.0)

provider_calls_path = out/'provider_calls.json'
write_json(provider_calls_path, results)
add_artifact(manifest, provider_calls_path, 'call_index', meta={'calls': len(results)})
for rr in results:
    add_artifact(manifest, Path(rr['path']), 'raw_provider_json', meta={'provider': rr['provider'], 'site_id': rr['site_id'], 'http_status': rr['status']})

manifest_path = out/'manifest.json'
write_json(manifest_path, manifest)

print('Calls:', len(results))
for rr in results:
    print('-', rr['provider'], rr['site_id'], rr['status'])
print('Wrote:', provider_calls_path)
print('Wrote:', manifest_path)
